In [ ]:
import shutil, os

drive_path = '/content/drive/MyDrive/TCC-TAKEO'

# Copiar extractor.py do Drive para o diretório local
src = f'{drive_path}/extractor.py'
if os.path.exists(src):
    shutil.copy2(src, 'extractor.py')
    print(f"Copiado {src} → extractor.py")

from extractor import *

# Etapa 1 — Extração de Notícias do InfoMoney

Este notebook executa a coleta de artigos de notícias do portal InfoMoney para três ações da B3.

## Fonte de Dados

- **API:** WordPress REST API (`/wp-json/wp/v2/posts`) do InfoMoney
- **Endpoint:** `https://www.infomoney.com.br/wp-json/wp/v2/posts`
- **Autenticação:** Nenhuma (API pública)
- **Paginação:** 100 artigos por página, ordenados por `modified` (desc)

## Módulo: `extractor.py`

| Componente | Descrição |
|------------|-----------|
| `Artigo` (dataclass) | Representa um artigo com campos: id, date, title, content, excerpt, keywords, etc. |
| `ExtratorDeNoticias` | Extrai artigos de um ticker em um período com retentativas (3x, backoff linear) |
| `extrair_varias_acoes()` | Extração paralela via `ThreadPoolExecutor` |
| `carregar_existentes()` | **[US-010]** Append incremental — carrega JSON existente e deduplicada por `id` |

## Pré-processamento

- `_strip_html()`: remove tags HTML e decodifica entidades (`&amp;` → `&`)
- `_normalize_text()`: normaliza whitespace e trunca textos longos
- `_extract_keywords()`: extrai keywords do schema Yoast SEO (JSON-LD)

## Volumes Coletados

| Ativo | Artigos (1ª coleta) | Artigos (expandido) |
|-------|---------------------|---------------------|
| PETR4 | ~1.775 | expandir com `incremental=True` |
| VALE3 | ~1.525 | expandir com `incremental=True` |
| ITUB4 | ~798 | 2.572 (5 anos) |

## Modo Incremental (US-010)

Para expandir o histórico sem perder dados existentes:
```python
extratores = extrair_varias_acoes(
    acoes=["PETR4", "ITUB4", "VALE3"],
    meses_atras=12 * 17,  # ~17 anos (desde 2009)
    incremental=True,      # carrega existentes e faz append
)
```

**Formatos de saída:** JSON (`{ticker}_noticias.json`) e CSV (`{ticker}_noticias.csv`)

In [ ]:
# Extrair multiplas acoes em PARALELO (multithreading) — período estendido (17 anos)
acoes = ["PETR4", "ITUB4", "VALE3"]

extratores = extrair_varias_acoes(
    acoes=acoes,
    meses_atras=12 * 17,  # ~17 anos (desde 2009)
    incremental=True,      # carrega existentes e faz append sem duplicatas
    max_workers=3,
)

In [ ]:
# Carregar notícias existentes do Drive para o diretório local
# (garante que o modo incremental tenha acesso aos dados já coletados)
import shutil, os

drive_path = '/content/drive/MyDrive/TCC-TAKEO'
acoes = ["PETR4", "ITUB4", "VALE3"]

for ticker in acoes:
    src = f'{drive_path}/{ticker.lower()}_noticias.json'
    dst = f'{ticker.lower()}_noticias.json'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"Copiado {src} → {dst}")
    elif os.path.exists(dst):
        print(f"{dst} já existe localmente")
    else:
        print(f"{src} não encontrado no Drive (será criado do zero)")

In [ ]:
# Salvar CSV local
extratores['PETR4'].salvar_csv('petr4_noticias.csv')
extratores['VALE3'].salvar_csv('vale3_noticias.csv')
extratores['ITUB4'].salvar_csv('itub4_noticias.csv')

# Salvar JSON e CSV no Google Drive
import shutil, os

drive_path = '/content/drive/MyDrive/TCC-TAKEO'
os.makedirs(drive_path, exist_ok=True)

for ticker in acoes:
    t = ticker.lower()
    for ext in ['json', 'csv']:
        src = f'{t}_noticias.{ext}'
        if os.path.exists(src):
            shutil.copy2(src, f'{drive_path}/{src}')

print(f"Arquivos salvos em {drive_path}/")
os.listdir(drive_path)

In [14]:
import os

In [29]:
os.listdir('drive/MyDrive')

['Resumo para a prova de GEO 31 10.gdoc',
 'Pasta ',
 'Reelaboracado de historia 1 bimestre.gdoc',
 ' reelaboracao de historia 2bi.gdoc',
 'Matérias',
 '3º bimestre',
 'Relaoração de hist bloco 1.gdoc',
 'capa relab bloco 1.gdoc',
 '20170406_122504.jpg',
 'Cópia de 20170406_122504.jpg',
 'Estrofes - Os Lusíadas.gdoc',
 'CANTO 4.gdoc',
 'Documento sem título (12).gdoc',
 'Trabalhos Poliedro 1° ano',
 'Trabalho de Gramática 20 03.gslides',
 'Documento sem título (11).gdoc',
 'dispensa.gdoc',
 'Trabalho_Motoristas_Final - Agosto - 2019-LLW-PC.gsheet',
 'Planilha sem título (4).gsheet',
 'Trabalho do Zezo.gslides',
 'Trabalho do Zezo.gdoc',
 'Documento sem título (10).gdoc',
 'Contaminação por radiação (1).pptm.gslides',
 'mariozao.jfif',
 'WhatsApp Image 2020-03-31 at 13.01.48.jpeg',
 'mariozão (2).jfif',
 'erickson (1).jfif',
 'walneide.jfif',
 'WhatsApp Image 2020-04-13 at 09.20.00.jpeg',
 'ft mariao.jpg',
 'mariozão (1).jfif',
 'erickson (2).jpg',
 'mariao (1).jpg',
 'e